# Linking elements together

As explained in the Data Structure chapter, `momepy` relies on links between different morphological elements. Each element has an index, and each of the small-scale elements also needs to know the index of the relevant higher-scale element. The case of block index is explained in the previous chapter, `momepy.generate_blocks` generates it together with blocks gdf.

## Getting the index of the street network

This notebook will explore how to link street network, both nodes and edges, to buildings and tessellation.

### Edges

For linking street network edges to buildings (or tessellation or other elements), `momepy` offers `momepy.get_nearest_street`. It simply returns a `Series` of indices of nearest streets for analysed gdf.

In [ ]:
import geopandas as gpd
import momepy

For illustration, we can use `bubenec` dataset embedded in `momepy`.

In [ ]:
path = momepy.datasets.get_path("bubenec")
buildings = gpd.read_file(path, layer="buildings")
streets = gpd.read_file(path, layer="streets")
tessellation = gpd.read_file(path, layer="tessellation")

Then we can link it to buildings. The only argument we might want to look at is `max_distance`, which is a maximum distance within which to query for nearest street. Note that for large areas, it is advised to set a limit to avoid long processing times.

In [ ]:
buildings["street_index"] = momepy.get_nearest_street(buildings, streets)

In [ ]:
ax = buildings.plot(
    column="street_index", categorical=True, cmap="tab20b", figsize=(8, 8)
)
streets.plot(ax=ax)
ax.set_axis_off()

Note: colormap does not have enough colours, that is why everything on the top-left looks the same. It is not.

### Nodes

The situation with nodes is slightly more complicated as you usually don't have or even need nodes. However, `momepy` includes some functions which are calculated on nodes (mostly in `graph` module). For that reason, we will pretend that we follow the usual workflow:

1. Street network `GeoDataFrame` (edges only)
2. networkx `Graph`
3. Street network - edges and nodes as separate `GeoDataFrames`.

In [ ]:
graph = momepy.gdf_to_nx(streets)

Some [graph-based analysis](../graph/graph.rst) happens here.

In [ ]:
nodes, edges = momepy.nx_to_gdf(graph)

In [ ]:
ax = edges.plot(
    column=edges.index.values, categorical=True, cmap="tab20b", figsize=(8, 8)
)
nodes.plot(ax=ax, zorder=2)
ax.set_axis_off()

For attaching node ID to buildings, we will need both, nodes and edges. Given the index `edges` created from the graph likely do not match the `streets`, determine which edge building belongs to using `momepy.get_nearest_street` and then find out which end of the edge is the closer one.

In [ ]:
buildings["edge_index"] = momepy.get_nearest_street(buildings, edges)

The node index is also inlcuded in `edges` as well, denoting each end of edge. (Length of the edge is also present as it was necessary to keep as an attribute for the graph.)

In [ ]:
edges.head()

We can now use `momepy.get_nearest_node` to attach nodes. Keep in mind that this is not just a nearest neighbor join, each nearest node is ensured to be on one end of the nearest edge.

In [ ]:
buildings["node_index"] = momepy.get_nearest_node(
    buildings, nodes, edges, buildings["edge_index"]
)

In [ ]:
ax = buildings.plot(
    column="node_index", categorical=True, cmap="tab20b", figsize=(8, 8)
)
nodes.plot(ax=ax, zorder=2)
edges.plot(ax=ax, zorder=1)
ax.set_axis_off()

### Transfer IDs to tessellation
All IDs are now stored in buildings gdf. We can copy them to tessellation directly as both share the same index.

In [ ]:
tessellation[["edge_index", "node_index"]] = buildings[
    ["edge_index", "node_index"]
]
tessellation.head()

Now we should be able to link all elements together as needed for all types of morphometric analysis in `momepy`.